# How to download a dataset and construct a dataframe

This notebook has developed by members of the [Language Technologies Lab](https://nlp.unibo.it), as a learning resource for the [Natural Language Processing course](https://www.unibo.it/en/study/phd-professional-masters-specialisation-schools-and-other-programmes/course-unit-catalogue/course-unit/2026/530664) of the [International Master's Degree in Artificial Intelligence](https://corsi.unibo.it/2cycle/artificial-intelligence/) of the [University of Bologna](https://www.unibo.it/en/homepage). It is distributed on [Virtuale](https://virtuale.unibo.it/) and can be downloaded personal use only. Any sharing or distribution of the notebook or its adaptations without written consent of the authors is forbidden.

Authors:
- Federico Ruggeri
- Andrea Galassi
- Eleonora Mancini
- Paolo Torroni

In [1]:
# preliminaries
import urllib
import requests
import tarfile
import pandas as pd
from tqdm import tqdm
from pathlib import Path

## Download dataset
We download the IMDB movie reviews dataset.
We also define a base folder for saving the dataset we download and its processing - to avoid doing it all over again each time we need it. You may change here the path from /Users/paolo/PycharmProjects/Datasets to whatever fits your file system organization.

In [2]:
local_folder = Path('/Users/paolo/PycharmProjects')
data_folder = local_folder.joinpath('Data')
print(f"Datasets directory: {data_folder}")

dataset_name = "aclImdb"
dataset_path = data_folder.joinpath(dataset_name)
if not dataset_path.exists():
    dataset_path.mkdir(parents=True)

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_url(download_path: Path, url: str):
    with DownloadProgressBar(unit='B', unit_scale=True,
                             miniters=1, desc=url.split('/')[-1]) as t:
        urllib.request.urlretrieve(url, filename=download_path, reporthook=t.update_to)


def download_dataset(download_path: Path, url: str):
    print("Downloading dataset...")
    download_url(url=url, download_path=download_path)
    print("Download complete!")


def extract_dataset(download_path: Path, extract_path: Path):
    print("Extracting dataset... (it may take a while...)")
    with tarfile.open(download_path) as loaded_tar:
        loaded_tar.extractall(extract_path)
    print("Extraction completed!")


url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
dataset_name = "aclImdb"

dataset_tar_path = dataset_path.joinpath(dataset_name + ".tar.gz")

if not dataset_tar_path.exists():
    download_dataset(dataset_tar_path, url)

dataset_folder = dataset_path.joinpath("Dataset")
if not dataset_folder.exists():
    print(f"Extracting {dataset_name} to {dataset_folder}")
    dataset_folder.mkdir(parents=True)
    extract_dataset(dataset_tar_path, dataset_folder)
else:
    print("Dataset folder already exists")

Datasets directory: /Users/paolo/PycharmProjects/Data
Dataset folder already exists


Show an example



In [3]:
# an example
sample_file_path = dataset_folder.joinpath(dataset_name,
                                    'train',
                                    'pos',
                                    "0_9.txt")

if sample_file_path.is_file():
    # open the file
    with sample_file_path.open(mode='r', encoding='utf-8') as text_file:
        text = text_file.read()
        print(text)

Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!


## Encode the dataset into a pandas.DataFrame

We encode the loaded dataset into a dataframe to better access to its content. In the end, we serialize and save the dataframe to file using pickle. If this has already been done before, we simply load the dataframe from the pickle archive.

In [4]:
df = pd.DataFrame()

dataframe_folder = dataset_path.joinpath("Dataframe")
dataframe_path = dataframe_folder.joinpath(dataset_name + ".pkl")

if dataframe_path.exists():
    #recover from pickle file
    print("recovering from pickle file", dataframe_path)
    df=pd.read_pickle(dataframe_path)
else:
    print("pickle file doesn't exist: creating dataframe")

    # read the dataset from file system and save it in a dataframe_rows list
    dataframe_rows = []
    for split in ['train', 'test']:
        for sentiment in ['pos', 'neg']:
            folder = dataset_folder.joinpath(dataset_name, split, sentiment)
            for sample_file_path in folder.glob('*.txt'):
                with sample_file_path.open(mode='r', encoding='utf-8') as text_file:
                    text = text_file.read()
                    score = sample_file_path.stem.split("_")[1]
                    score = int(score)
                    file_id = sample_file_path.stem.split("_")[0]

                    num_sentiment = 1 if sentiment == 'pos' else 0

                    dataframe_row = {
                        "file_id": file_id,
                        "score": score,
                        "sentiment": num_sentiment,
                        "split": split,
                        "text": text
                    }

                    dataframe_rows.append(dataframe_row)

    # transform dataframe_rows in a proper dataframe
    df = pd.DataFrame(dataframe_rows)
    df = df[["file_id",
             "score",
             "sentiment",
             "split",
             "text"]
           ]
    #save to file in pickle format
    dataframe_folder.mkdir(parents=True)
    df.to_pickle(dataframe_path)
    print("done")

recovering from pickle file /Users/paolo/PycharmProjects/Data/aclImdb/Dataframe/aclImdb.pkl
